In [1]:
import sqlite3
import re


In [2]:
import pandas as pd

df_trucks = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Trucks")
df_trucks.rename(columns={"Truck Number": "truck_number", "Driver Name": "driver_name","Trucking Company": "trucking_company", "License Plate": "license_plate"}, inplace=True)
df_trucks = df_trucks.iloc[1:].reset_index(drop=True)


df_customers = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Customers")
df_customers['zip_code_str'] = df_customers['Zip Code'].apply(lambda x: str(int(x)) if pd.notnull(x) else '')
parts = ["Address Line 1","Address Line 2","City","State","zip_code_str",]
df_customers[parts] = df_customers[parts].fillna("").astype(str)
df_customers["full_Address"] = (df_customers[parts].agg(", ".join, axis=1).str.replace(r",\s*,", ",", regex=True).str.strip(", "))


df_materials = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Material")
df_materials['5']=df_materials['Rev# 11\n4/5 Axle']
df_materials.rename(columns={'Rev# 11\n1 Axle':'1', 'Rev# 11\nTandem':'2', 'Rev# 11\nTriAxle':'3', 'Rev# 11\n4/5 Axle':'4', 'Rev# 11\n6 Axle':'6', 'Rev# 11\nSemi':'7', 'Rev# 11\nHydVac':'8','Rev# 11\nDIRT IN':'9'}, inplace=True)

In [6]:
try:
    cursor.close()
except:
    pass
try:
    conn.close()
except:
    pass

In [5]:
conn = sqlite3.connect("data/tickets.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()


for index, row in df_trucks.iterrows():
    axle =row["Size"]
    truck_size_label = f"axle{axle}"
    cursor.execute("""
        INSERT INTO trucks_main (truck_number, trucking_company, notes, truck_size, phone, license_plate)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row['truck_number'], row['trucking_company'], row['Notes'], truck_size_label, row['Phone'], row['license_plate']))

conn.commit()
print("Data inserted successfully into trucks_main!")

for index, row in df_materials.iterrows():
    cursor.execute("""
        INSERT INTO material_price (cat, material, axle1, axle2, axle3, axle4, axle5, axle6, axle7, axle8, axle9)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (row['Cat'], row['Material new name'], row['1'], row['2'], row['3'], row['4'], row['5'], row['6'], row['7'], row['8'], row['9']))

conn.commit()
print("Data inserted successfully into material_price!")

for index, row in df_customers.iterrows():
    cursor.execute("""
        INSERT INTO customers (customer_name, full_address, contact_person, phone_number, notes)
        VALUES (?, ?, ?, ?, ?)
    """, (row['Customer Name'], row['full_Address'], row['Contact'], row['Phone #'], row['Notes']))

conn.commit()
print("Data inserted successfully into customers!")


try:
    cursor.close()
except:
    pass
try:
    conn.close()
except:
    pass


Data inserted successfully into trucks_main!


IntegrityError: NOT NULL constraint failed: material_price.direction

In [ ]:
conn = sqlite3.connect("data/tickets.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("SELECT * FROM material_price LIMIT 5")
for row in cursor.fetchall():
    print(dict(row))
try:
    cursor.close()
except:
    pass
try:
    conn.close()
except:
    pass


{'id': 1, 'cat': 8, 'material': 'Broken Asphalt (IN)', 'axle1': 125.0, 'axle2': 150.0, 'axle3': 200.0, 'axle4': 275.0, 'axle5': 275.0, 'axle6': 300.0, 'axle7': 325.0, 'axle8': None, 'axle9': None, 'active': 1}
{'id': 2, 'cat': 4, 'material': 'Broken Concrete (IN) ', 'axle1': 75.0, 'axle2': 90.0, 'axle3': 120.0, 'axle4': 165.0, 'axle5': 165.0, 'axle6': 180.0, 'axle7': 195.0, 'axle8': None, 'axle9': None, 'active': 1}
{'id': 3, 'cat': 3, 'material': 'Bulk Topsoil (clean) (IN)', 'axle1': 40.0, 'axle2': 48.0, 'axle3': 64.0, 'axle4': 88.0, 'axle5': 88.0, 'axle6': 96.0, 'axle7': 104.0, 'axle8': None, 'axle9': None, 'active': 1}
{'id': 4, 'cat': 12, 'material': 'Bridge Deck (IN)', 'axle1': 625.0, 'axle2': 750.0, 'axle3': 1000.0, 'axle4': 1375.0, 'axle5': 1375.0, 'axle6': 1500.0, 'axle7': 1625.0, 'axle8': None, 'axle9': None, 'active': 1}
{'id': 5, 'cat': 1, 'material': 'Asphalt Grindings (no chunks) (IN)', 'axle1': 20.0, 'axle2': 24.0, 'axle3': 32.0, 'axle4': 44.0, 'axle5': 44.0, 'axle6': 48.